# Variant C2: keep the frequency rows

**One architecture change, one run, one pre-registered verdict.**

Today the STFT branch ends with `f.mean(dim=2)`. It holds a block of 64 pattern-detectors x 8 frequency bands x 62 moments and **averages the 8 bands into a single number**. So "a wide jammer block on bands 1-3 *and* a thin hopper dash on band 6" melts into nearly the same number as "a slightly louder jammer". When a jammer sits on top of FHSS the model reports JAMMING and forgets the hopper underneath.

**C2** replaces that average with a small learned layer that **keeps all 8 bands** (64 x 8 = 512 numbers per moment, mapped back to 64). The output width is unchanged, so the rest of the network is untouched. Cost: +32,960 parameters (148,938 -> 181,898). Nothing else about the model, the data, or the training recipe changes.

**The goal:** FHSS must go **up** where the jammer masks it, **without** JAMMING moving. Both are judged automatically at the end.

### Time
About **2.5 hours** to train on a T4, then about 10 minutes to evaluate. Run with `SEED = 2000` first. A single training run wobbles by up to about 6 points, so **confirm a PASS by running again with `SEED = 2001`**.

### Before you start
Push the branch once from your laptop, or Colab cannot clone it:
```
git push -u origin c2-keep-stft-rows
```

### How to read the result
Thresholds are recalibrated for every model so each judged class keeps about 83% recall on validation. So a better model does **not** show up as a higher headline recall. It shows up as **higher precision**, and as better recall *in company* (FHSS with a jammer present). That is exactly what the checks below measure.

### The baseline to beat
Member 0 of the shipped ensemble, the same seed 2000, thresholds recalibrated the same way (files in `docs/experiments/c2_baseline_*.json`).

| Group | Check | Baseline | Needs |
|---|---|---|---|
| **FHSS goes up** | FHSS recall with a jammer present (test split) | 55.4% | at least **+10 points** |
| | jammed FHSS reported as **both** classes, +6 dB | 28.3% | at least **55%** |
| | jammed FHSS reported as **both** classes, +10 dB | 19.0% | at least **45%** |
| | FHSS recall at JSR +10 dB (the team's pre-registered bar) | 2.5% | above **25%** |
| | FHSS precision (no false-alarm blow-up) | 54.3% | not more than 3 points lower |
| **JAMMING intact** | JAMMING recall | 84.5% | at least 80% and not >3 points lower |
| | JAMMING precision | 98.8% | not more than 2 points lower |
| | false alarm rate on civilian-only windows | 0.08% | at most **0.1%** |
| | comms-vs-jamming accuracy | 97.0% | not more than 1 point lower |
| | jammer with no victim wrongly called FHSS | 1.8% | not more than 3 points higher |
| | jammer **missed** (FHSS reported without JAMMING), +6 / +10 dB | 3.3% / 5.0% | not more than 3 points higher |
| **Nothing else breaks** | LFM_RADAR and FHSS recall | 84.7% / 82.6% | at least 80% |
| | LFM_RADAR precision | 51.4% | not more than 3 points lower |
| | standalone FHSS recall, SNR >= -6 dB | about 99% | at least 97% |

## 1. GPU on

**Runtime -> Change runtime type -> T4 GPU -> Save**, before anything else. Changing it later restarts the machine and wipes whatever you had loaded.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY -- change the runtime type, then run this again')

## 2. Code

In [ ]:
BRANCH = 'c2-keep-stft-rows'      # the branch you pushed

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -4
!pip install -q pyyaml h5py scikit-learn
!ls docs/experiments/c2_baseline_*.json

## 3. Data

The 128,400-window dataset (`X.npy`, `y.npy`, `snr_labels.npy`) from your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Find X.npy anywhere in My Drive, so you don't have to guess the path.
import subprocess
hits = subprocess.run(['find', '/content/drive/MyDrive', '-name', 'X.npy'],
                      capture_output=True, text=True).stdout.split()
for h in hits:
    print(h)
print('\n-> set DRIVE_DATA below to the FOLDER holding the copy you want')
if not hits:
    print('nothing found: the data is probably in a computer mirror, not My Drive')

In [ ]:
import shutil, pathlib
import numpy as np

DRIVE_DATA = '/content/drive/MyDrive/sedicAI_NEXA/data/processed'   # EDIT ME

dest = pathlib.Path('/content/sedicAI_NEXA/data/processed')
dest.mkdir(parents=True, exist_ok=True)
for name in ('X.npy', 'y.npy', 'snr_labels.npy'):
    src = pathlib.Path(DRIVE_DATA) / name
    assert src.exists(), f'not found: {src} -- fix DRIVE_DATA above'
    shutil.copy(src, dest / name)
    print(f'{name:16s} {(dest / name).stat().st_size / 1e6:8.1f} MB')

X = np.load(dest / 'X.npy', mmap_mode='r')
y = np.load(dest / 'y.npy'); snr = np.load(dest / 'snr_labels.npy')
assert X.shape == (128400, 2, 512), f'unexpected X shape {X.shape}: this must be the rebuilt 128,400-window dataset'
assert y.shape == (128400, 8) and snr.shape == (128400,), (y.shape, snr.shape)
print('X', X.shape, ' y', y.shape, ' snr bins', sorted(set(snr.tolist())))

## 4. Choose the run

`RUN = 'c2'` is the test this notebook is built for. Every other cell of the experiment plan is one word away, so you can reuse the notebook (run each in a fresh session, or one after another).

| `RUN` | keeps the rows | hand-built side channels | low-SNR sampling | what it tests |
|---|---|---|---|---|
| **`c2`** | yes | no | today's (10x) | **the architecture change alone** |
| `c2_a` | yes | no | softer (3x) | C2 plus training exposure |
| `a` | no | no | softer (3x) | training exposure alone |
| `flag` | no | yes | today's | Eileen's `stft_freq_summary` features |
| `c2_flag` | yes | yes | today's | both architecture changes |
| `c2_flag_a` | yes | yes | softer | everything at once |
| `baseline` | no | no | today's | a fresh re-run, which measures run-to-run noise |

In [ ]:
RUNS = {
    'c2':        dict(keep_rows=True,  freq_summary=False, divisor=20),
    'c2_a':      dict(keep_rows=True,  freq_summary=False, divisor=40),
    'a':         dict(keep_rows=False, freq_summary=False, divisor=40),
    'flag':      dict(keep_rows=False, freq_summary=True,  divisor=20),
    'c2_flag':   dict(keep_rows=True,  freq_summary=True,  divisor=20),
    'c2_flag_a': dict(keep_rows=True,  freq_summary=True,  divisor=40),
    'baseline':  dict(keep_rows=False, freq_summary=False, divisor=20),
}
RUN  = 'c2'      # <- the only thing to change
SEED = 2000      # then 2001 to confirm a PASS

import sys
sys.path.insert(0, '/content/sedicAI_NEXA')
from scripts.run_stft_experiment import run_tag      # the runner names its files with this same function

cfg = RUNS[RUN]
TAG = run_tag(cfg['freq_summary'], cfg['keep_rows'], cfg['divisor'], SEED)

TRAIN_ARGS = f"--seed {SEED} --divisor {cfg['divisor']}"
TRAIN_ARGS += '' if cfg['freq_summary'] else ' --no-freq-summary'
TRAIN_ARGS += ' --keep-rows' if cfg['keep_rows'] else ''
MODEL_FLAGS = ('--stft-keep-rows ' if cfg['keep_rows'] else '') + ('--stft-freq-summary' if cfg['freq_summary'] else '')

print('run           :', RUN, cfg)
print('files         : results/experiment_%s.pt (+ history, config)' % TAG)
print('train command : python scripts/run_stft_experiment.py', TRAIN_ARGS)
print('model flags   :', MODEL_FLAGS or '(none)')

### Prove the switches took, before spending 2.5 hours

Builds the model with your choice and checks its size. The shipped model has **148,938** parameters; C2 must show **181,898**.

In [ ]:
from src.config import CFG, CLASSES
CFG.setdefault('model', {})['stft_freq_summary'] = cfg['freq_summary']
CFG['model']['stft_keep_rows'] = cfg['keep_rows']
from src.models.amc_cnn import AMC_CNN

m = AMC_CNN(num_classes=len(CLASSES), input_len=512)
n = sum(p.numel() for p in m.parameters())
print(f'parameters {n:,}   keep_rows={m.stft_branch.keep_rows}   freq_summary={m.stft_branch.freq_summary}')
assert m.stft_branch.keep_rows == cfg['keep_rows'] and m.stft_branch.freq_summary == cfg['freq_summary']
expected = {'c2': 181898, 'c2_a': 181898, 'a': 148938, 'baseline': 148938, 'flag': 149709,
            'c2_flag': 215437, 'c2_flag_a': 215437}[RUN]
assert n == expected, f'expected {expected:,} parameters for {RUN}, got {n:,} -- stop, do not train'
print('OK: this is the architecture you asked for')
del m

## 5. Train

About 2.5 hours on a T4. The best checkpoint is rewritten **every time validation improves**, and a per-epoch history file every epoch, so a dropped connection loses at most the current epoch.

The next cell starts a background saver that copies both to your Drive every 5 minutes. **Run it first**, then start training.

In [ ]:
import threading, time, shutil, pathlib
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/sedicAI_NEXA_experiments')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def _autosave():
    while True:
        time.sleep(300)
        for f in pathlib.Path('/content/sedicAI_NEXA/results').glob(f'experiment_{TAG}*'):
            try:
                shutil.copy(f, DRIVE_OUT / f.name)
            except Exception as e:
                print('autosave skipped', f.name, e)

threading.Thread(target=_autosave, daemon=True).start()
print('autosaving results/experiment_%s* to' % TAG, DRIVE_OUT, 'every 5 minutes')

In [ ]:
import time
t0 = time.time()
!python -u scripts/run_stft_experiment.py {TRAIN_ARGS}
print(f'\ntraining took {(time.time() - t0) / 3600:.2f} h')

In [ ]:
import json, pathlib, shutil
ckpt = pathlib.Path(f'results/experiment_{TAG}.pt')
hist = pathlib.Path(f'results/experiment_{TAG}_history.json')
assert ckpt.exists(), 'no checkpoint: did training finish?'
h = json.load(open(hist))
print(f"epochs run {len(h['epochs'])} of {h['epochs_planned']}   best epoch {h['best_epoch']}   best val loss {h['best_val_loss']:.4f}")
for f in pathlib.Path('results').glob(f'experiment_{TAG}*'):
    shutil.copy(f, DRIVE_OUT / f.name)
print('copied to Drive:', DRIVE_OUT)

## 6. Evaluate everything

Recalibrates this model's thresholds on the **validation** split (the team's own rule: the highest threshold that keeps recall at 83%), then scores the **test** split and compares with the baseline.

You get, for all 8 classes: **precision, recall, F1, accuracy, balanced accuracy, specificity, support, and the TP / FP / FN / TN counts**. Also the benchmark gate, comms-vs-jamming (accuracy, jamming recall, false alarm rate), coarse-tier accuracy, dense-QAM recall, the radar/FHSS confusion, recall **in company** (alone / with another emitter / with a jammer), and recall by SNR.

In [ ]:
!python scripts/evaluate_experiment.py --checkpoint results/experiment_{TAG}.pt {MODEL_FLAGS} --name {TAG} --baseline docs/experiments/c2_baseline_eval.json

## 7. Probes, with this model's own thresholds

Two probes generate fresh windows with a known answer:

- **`high_snr_probe`**: jammer laid over FHSS. Reports what the model says, in four columns: **both** (correct), **JAMMING only** (the failure being fixed), **FHSS only** (the jammer got missed, which must not grow), and **neither**.
- **`probe_jsr`**: sweeps the jammer strength from equal power to +20 dB (the team's pre-registered bar: FHSS recall at +10 dB above 25%).

In [ ]:
!python scripts/high_snr_probe.py --n 300 --class FHSS --seed 7 --checkpoint results/experiment_{TAG}.pt {MODEL_FLAGS} --thresholds results/thresholds_{TAG}.json --out results/high_snr_{TAG}.json

In [ ]:
!python scripts/probe_jsr.py --n 600 --seed 0 --checkpoint results/experiment_{TAG}.pt {MODEL_FLAGS} --thresholds results/thresholds_{TAG}.json --out results/jsr_{TAG}.json

## 8. The verdict

Applies the checks from the table at the top. **PASS means: FHSS went up where it is masked, and JAMMING (and everything else) held.** Any single failed check is a FAIL, and it names which one.

In [ ]:
!python scripts/evaluate_experiment.py --verdict-only --eval-json results/eval_{TAG}.json --baseline docs/experiments/c2_baseline_eval.json --high-snr results/high_snr_{TAG}.json --baseline-high-snr docs/experiments/c2_baseline_high_snr.json --jsr results/jsr_{TAG}.json --baseline-jsr docs/experiments/c2_baseline_jsr.json

## 9. Pictures

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt

R, E = pathlib.Path('results'), pathlib.Path('docs/experiments')
new_eval,  base_eval = json.load(open(R / f'eval_{TAG}.json')), json.load(open(E / 'c2_baseline_eval.json'))
new_high,  base_high = json.load(open(R / f'high_snr_{TAG}.json')), json.load(open(E / 'c2_baseline_high_snr.json'))
hist = json.load(open(R / f'experiment_{TAG}_history.json'))
CLS = ['BPSK', 'QPSK', '16QAM', '64QAM', 'LFM_RADAR', 'FHSS', 'JAMMING', 'NOISE_FLOOR']
snrs = sorted(base_eval['by_snr'], key=float)
BLUE, ORANGE = '#64748b', '#16a34a'

fig, ax = plt.subplots(2, 3, figsize=(21, 11))

# 1) what is reported for FHSS + a jammer
segs = [('both', '#16a34a', 'both (correct)'), ('jamming_only', '#dc2626', 'JAMMING only'),
        ('class_only', '#f59e0b', 'FHSS only (jammer missed)'), ('neither', '#cbd5e1', 'neither')]
xs, labels, k = [], [], 0
for s in ('2.0', '6.0', '10.0'):
    for name, d in (('base', base_high), ('new', new_high)):
        o = d['overlay_outcomes'][s]; bottom = 0
        for key, col, lab in segs:
            ax[0, 0].bar(k, o[key], bottom=bottom, color=col, label=lab if (s == '2.0' and name == 'base') else None)
            bottom += o[key]
        xs.append(k); labels.append(f"{'base' if name == 'base' else 'C2'}\n+{int(float(s))} dB"); k += 1
    k += 0.6
ax[0, 0].set_xticks(xs); ax[0, 0].set_xticklabels(labels); ax[0, 0].set_ylabel('% of jammed-FHSS windows')
ax[0, 0].set_title('Jammed FHSS: what the model reports\n(goal: less red, more green, orange must not grow)')
ax[0, 0].legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=4)

# 2) FHSS recall when a jammer is in the window, by SNR
def ctx(ev, cls):
    return [np.nan if ev['context_by_snr'][s][cls]['with_jammer']['recall'] is None
            else 100 * ev['context_by_snr'][s][cls]['with_jammer']['recall'] for s in snrs]
xv = [float(s) for s in snrs]
ax[0, 1].plot(xv, ctx(base_eval, 'FHSS'), 'o--', color=BLUE, label='baseline')
ax[0, 1].plot(xv, ctx(new_eval, 'FHSS'), 'o-', color=ORANGE, lw=2.5, label='C2')
ax[0, 1].set_title('FHSS recall WHEN A JAMMER IS PRESENT (test split)'); ax[0, 1].set_xlabel('SNR (dB)')
ax[0, 1].set_ylabel('recall (%)'); ax[0, 1].legend(); ax[0, 1].grid(alpha=0.3)

# 3) judged-class recall by SNR
for cls, ls in (('FHSS', 'o'), ('JAMMING', 's')):
    for ev, style, col, lab in ((base_eval, '--', BLUE, 'baseline'), (new_eval, '-', ORANGE, 'C2')):
        ax[0, 2].plot(xv, [100 * ev['by_snr'][s][cls]['recall'] for s in snrs], ls + style, color=col,
                      alpha=0.9, label=f'{cls}, {lab}')
ax[0, 2].axhline(80, color='red', ls=':', lw=1); ax[0, 2].set_title('Recall by SNR: FHSS up, JAMMING must not move')
ax[0, 2].set_xlabel('SNR (dB)'); ax[0, 2].set_ylabel('recall (%)'); ax[0, 2].legend(fontsize=8); ax[0, 2].grid(alpha=0.3)

# 4) precision and 5) recall per class
w = 0.38
for a, key, title in ((ax[1, 0], 'precision', 'Precision per class (at calibrated thresholds)'),
                      (ax[1, 1], 'recall', 'Recall per class (pinned near 83% by calibration)')):
    a.bar(np.arange(8) - w / 2, [100 * base_eval['per_class'][c][key] for c in CLS], w, color=BLUE, label='baseline')
    a.bar(np.arange(8) + w / 2, [100 * new_eval['per_class'][c][key] for c in CLS], w, color=ORANGE, label='C2')
    a.set_xticks(range(8)); a.set_xticklabels(CLS, rotation=35, ha='right'); a.set_title(title); a.legend()
    a.set_ylim(0, 105)

# 6) training curves
ep = [e['epoch'] for e in hist['epochs']]
ax[1, 2].plot(ep, [e['train_loss'] for e in hist['epochs']], label='train loss')
ax[1, 2].plot(ep, [e['val_loss'] for e in hist['epochs']], label='validation loss')
ax[1, 2].axvline(hist['best_epoch'], color='grey', ls=':'); ax[1, 2].set_xlabel('epoch')
ax[1, 2].set_title(f"Training: best epoch {hist['best_epoch']}"); ax[1, 2].legend(); ax[1, 2].grid(alpha=0.3)

fig.suptitle(f'C2 (keep the frequency rows) vs baseline   -   run {TAG}', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig(R / f'c2_summary_{TAG}.png', dpi=110); plt.show()

## 10. Save everything to Drive before the tab closes

In [ ]:
import shutil, pathlib
keep = [f'experiment_{TAG}.pt', f'experiment_{TAG}_history.json', f'experiment_{TAG}_config.json',
        f'eval_{TAG}.json', f'thresholds_{TAG}.json', f'high_snr_{TAG}.json', f'jsr_{TAG}.json', f'c2_summary_{TAG}.png']
for name in keep:
    f = pathlib.Path('results') / name
    if f.exists():
        shutil.copy(f, DRIVE_OUT / f.name); print('saved to Drive:', f.name)
    else:
        print('MISSING (did that step run?):', name)

## What to do next

**If the verdict is PASS**
1. Run it again with `SEED = 2001` (section 4). Two passes in a row means it is real, not luck.
2. Then it earns a full 5-model ensemble retrain, recalibration, and a fresh evaluation. Set `stft_keep_rows: true` in `configs/default.yaml` for that, and re-export the web model with `scripts/export_onnx.py`. Every number in the report (sections 3, 6, 7, 8) must be regenerated, and the architecture description (parameter count 148,938 -> about 181,898) must change.
3. Try `RUN = 'c2_a'` to see whether softer low-SNR sampling adds to it.

**If the verdict is FAIL**, read which group failed:
- *"FHSS goes up" failed, guardrails passed*: C2 alone is not enough. Try `c2_a` (training exposure), then `c2_flag_a`.
- *A JAMMING guardrail failed*: FHSS and JAMMING are trading boundary space, which is a known trap for this pair (recall rose 82.5 -> 92.2 while JAMMING fell 80.0 -> 67.5 in an earlier round). Do **not** ship it.
- *Both failed*: the change hurt. Check the training curves for overfitting (validation loss rising while train loss falls).

**Rules of the road**
- One run is one sample. A single run varies by up to about 6 points, which is why the FHSS gain is required to be large (+10 points), not marginal.
- The 5 shipped models and their thresholds are not touched by any of this.
- Thresholds must be recalibrated for any retrained model. This notebook does that automatically, but the final submission's `configs/default.yaml` needs the new values pasted in by hand.